# Catalogo de scripts para SMTC
- V1. Version inicial

In [57]:
# Catalogo
from github import Github
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML
from dotenv import load_dotenv
import os
load_dotenv()
token = os.getenv('GITHUB_TOKEN')

if token is None:
    raise ValueError("GitHub token not found. Please ensure it is set in the .env file.")

# Authenticate to GitHub
g = Github(token)

repo_name = "ggteck/SMTC"

repo = g.get_repo(repo_name)

# List all scripts in the repository (assuming scripts have a .py extension)
contents = repo.get_contents("")




scripts = []
while contents:
    file_content = contents.pop(0)
    if file_content.type == "dir":
        contents.extend(repo.get_contents(file_content.path))
    else:
        if file_content.path.endswith('.ipynb'):
            scripts.append(file_content)

# Retrieve version history for each script
data = []
for script in scripts:
    commits = repo.get_commits(path=script.path)
    commit_info = [f"{commit.commit.message}" for commit in commits]
    data.append({
        'Script Name': script.path,
        'Version History': ', '.join(commit_info),
        'Script Path': script.path
    })

# Create DataFrame
df = pd.DataFrame(data)

# Define a function to create a download button for each script
def create_download_button(script_path):
    button = widgets.Button(description='Download Latest')
    def on_click(b):
        # Download the latest version of the script
        file_content = repo.get_contents(script_path)
        content = file_content.decoded_content.decode()
        # Save the script locally
        filename = script_path.split('/')[-1]
        with open(filename, 'w') as f:
            f.write(content)
        print(f"Downloaded {filename}")
    button.on_click(on_click)
    return button

# Add the download button to the DataFrame
df['Download'] = df['Script Path'].apply(create_download_button)

def display_dataframe(df):
    # Drop the 'Script Path' column for the display table
    display_df = df[['Script Name', 'Version History']].copy()
    
    # Add download buttons in the Download column
    display_df['Download'] = df['Download']
    
    # Create header widgets with grey background and borders
    header_widgets = [
        widgets.HTML(value=f"<b>{col}</b>", 
                     layout=widgets.Layout(border="1px solid black", background="lightgrey", padding="5px"))
        for col in display_df.columns[:2]
    ]
    # Add an empty widget for the third column header (Download), so it has no visible label
    header_widgets.append(
        widgets.HTML(value="", layout=widgets.Layout( background="lightgrey", padding="5px"))
    )
    
    # Create cell widgets with borders
    cell_widgets = [
        widgets.HTML(value=display_df.iloc[i, j], 
                     layout=widgets.Layout(border="1px solid black", padding="5px"))
        if j < 2 else display_df.iloc[i, j]  # Download button cell remains a widget
        for i in range(len(display_df)) for j in range(3)
    ]
    
    # Combine header and cell widgets into a grid
    table = widgets.GridBox(
        children=header_widgets + cell_widgets,
        layout=widgets.Layout(
            grid_template_columns="30% 50% 20%",
            grid_gap="0px"
        )
    )
    
    display(table)
# Display the DataFrame
display_dataframe(df)

GridBox(children=(HTML(value='<b>Script Name</b>', layout=Layout(border_bottom='1px solid black', border_left=…